# Paper 67 — Original Study Replication

**Enze Hu · Jinrui Hu | IEOR 4576 | Wood, Roberts & Zohren (2022)**

完整原论文复现 / Full original-study reproduction. Select a Python environment installed from `pyproject.toml`, then **Run All**. 默认 `FRESH_MODELS=True`：所有 LSTM / CPD-LSTM 候选从随机初始化训练，不加载旧权重；旧检查点自动归档。训练 cell 下实时更新训练/验证曲线。

The notebook invokes reusable source modules. It does not read `Paper67_Progress_Executed.ipynb`. Raw data and deterministic causal feature caches may be reused. On a new clone, missing raw data are downloaded through your own authorized WRDS account and missing features are computed. No fitted checkpoints are required. No final results are claimed before execution completes.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/momentum_lab').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from momentum_lab.paths import initialize_notebook_environment
initialize_notebook_environment()
from momentum_lab.reporting import conclusion_text
pd.options.display.max_rows = 60
pd.options.display.max_columns = 15
pd.options.display.float_format = '{:.4f}'.format
T = ROOT/'results/replication/tables'
F = ROOT/'results/replication/figures'
E = ROOT/'results/extension/tables'
print('Project:', ROOT)
print('Notebook kernel:', sys.executable)

## Run the complete local reproduction

设 `RUN_REPRODUCTION=True` 运行全部 50-candidate searches，再完成回测、交易成本、Exhibits 3/4、图和 PDF 报告。`FRESH_MODELS=True` 会归档旧权重并从零训练；仅在意外中断后想接着本次运行时改为 `False`。这一步使用本机 CPU/GPU，自动选择 CUDA、Metal 或 CPU；不调用模型 API，不需要助手在线。首次 clone 缺少数据时会使用你自己的 WRDS 授权下载。不要同时启动另一个训练命令。保持电脑唤醒；这一格长时间显示 `[*]` 是正常训练。日志保存在 `results/replication/logs/local_notebook_run.log`。训练中断后重新运行会保留已完成的 candidate；当时尚未完成的 candidate 会重跑。

Once all outputs exist, set `RUN_REPRODUCTION=False` to read them without training. After Run All finishes, **save the notebook** to retain every output.

In [ ]:
CONFIG = 'configs/config.yaml'
MODEL_WORKERS = 3
RUN_REPRODUCTION = True
FRESH_MODELS = True

if RUN_REPRODUCTION:
    from momentum_lab.notebook_runner import run_prepared
    manifest = run_prepared(CONFIG, model_workers=MODEL_WORKERS, fresh_models=FRESH_MODELS)
else:
    manifest = json.loads((ROOT/'report/execution_manifest.json').read_text())
    assert not manifest['missing_experiments'], 'Experiments are incomplete: set RUN_REPRODUCTION=True.'
print('Completed searches:', manifest['completed_experiments'], '/', manifest['expected_experiments'])
print('Candidate fits:', manifest['completed_trials'])
print('Daily CPD rows:', manifest['cpd_daily_rows'])

## Original methodology and implemented conventions

The study is Wood, Roberts & Zohren (2022), *Slow Momentum with Fast Reversion* ([paper](https://arxiv.org/abs/2105.13727)), not the Momentum Transformer study. The reference list's item number is inconsistent with the requested title; this reproduction follows the title and arXiv identifier.

For each instrument, positions are selected at close t and evaluated against t-to-t+1 returns. The raw futures panel rolls five business days before the earliest vendor first-notice/last-trade date, selects the nearest eligible maturity with an observed positive settlement, and computes each day's price change within the previously selected contract. Missing historical volume/OI is not treated as a market inception date. For GI, ZW and ZZ, missing composite expiry metadata is supplemented from the vendor's pit-contract calendar for the identical underlying and delivery month. No expiration is inferred from the last observed price. Missing settlements are valued causally at the last observed mark and explicitly counted; these marks are not assumed to be executable fills. An index compounded from these real returns is the input to the authors' preserved feature code.

The features are 1/21/63/126/252-session volatility-normalized returns and MACD pairs (8,24), (16,48), (32,96). The classical MACD baseline maps the mean of the three normalized trend estimates through the author's separate `scale_signal` function, phi(y)=y exp(−y²/4)/0.89, as in equation (7) of the [foundational DMN reference](https://arxiv.org/html/1904.04912). The neural MACD inputs remain untransformed. The author implementation winsorizes the price index at five EWM standard deviations with half-life 252; daily volatility uses a 60-session EWM span. The original-style evaluation uses that implementation's winsorized return labels. The extension evaluates the positions on raw, un-winsorized same-contract returns as a separate tradability diagnostic.

For each trailing l+1 observation window, GP-CPD fits a stationary Matérn 3/2 covariance and a sigmoid-weighted pair of Matérn kernels by L-BFGS-B (maximum 200 iterations). Windows are reinitialized independently, with all stationary parameters set to one; the fitted stationary kernel initializes the changepoint model. Severity is sigmoid(NLML_M − NLML_CP). Relative location follows the author's code, (t−c)/l, which is the complement of the formula printed in equation (10). The Numba Kalman likelihood is algebraically equivalent to the dense GP covariance. Numerical stabilization adds 1e-8 diagonal jitter and bounded log parameters.

The stateless LSTM has 63 time steps, tanh position outputs, input and output dropout, and the pooled negative Sharpe objective on asset-time observations. Training uses non-overlapping sequences, shuffled sequence order, Adam, parameter-wise gradient norm clipping, a maximum output weight norm of three, up to 300 epochs, and 25-epoch validation patience. The PyTorch port freezes its redundant recurrent bias to match Keras' single bias. Backend initialization and optimizer arithmetic need not be bitwise identical to TensorFlow.

Training expands from 1990, with per-instrument chronological 90/10 train/validation splits. Labels crossing a split boundary are purged. An instrument needs at least one 63-step validation sequence and must satisfy Appendix A's first-test eligibility. Test windows are 1995–1999, 2000–2004, 2005–2009, 2010–2014, and 2015–2019; 2020 is reported as a separate frozen one-year test after a pre-2020 refit. This resolves the manuscript's ambiguous '1995–2020' endpoint explicitly; the 2020 window is excluded from the five-window exhibit averages.

All five CPD horizons (10,21,63,126,252) and the no-CPD LSTM are fitted. The main configuration requests 50 distinct random draws from the full published grid per model/window, using automatic CUDA/Metal/CPU selection and pre-test pooled validation Sharpe for selection. The fresh notebook run trains every candidate from random initialization; older model checkpoints are archived and are not used. The paper specifies 50 random-search iterations; the supporting later author repository defaults to diversified validation Sharpe. The inventory and manifest state the executed search count; a configuration file is not evidence that its experiments have run. One complete search repetition is conducted, with base seed 67 and per-candidate seeds 67+trial. Repeated-seed experiments are not represented as executed. Optimized CPD means selecting the lowest validation loss among the five completed CPD models, independently for each test window.

Exhibits 3 and 4 average each metric across the five main test windows. Annual return is compounded CAGR; Sharpe uses arithmetic mean excess P&L and sample volatility. The portfolio uses equal 1/N weights across the fold's eligible instruments, with no-return dates contributing cash. Exhibit 4 follows the later author aggregation code: use one factor, 15% divided by the mean annualized volatility of the five main windows, on the whole daily return series, then recompute each window's metrics. This targets 15% mean window volatility, not 15% in every window. The rounded printed paper is also consistent with linear scaling of averaged dimensional metrics; that display diagnostic is saved separately, because linear scaling of CAGR or drawdown is not identical to recomputation on compounded returns. These ex-post multipliers are not investable risk forecasts. No Sharpe is assumed to improve.

The data substitute is WRDS LSEG/Datastream individual-contract history, not the proprietary Pinnacle CLC panel. Session conventions, roll construction, starting dates and source revisions differ. Appendix A's NR and ZR series remain distinct; discontinued NR is not backfilled. The Euro series has no fabricated pre-euro/Deutsche-Mark extension. Several other markets start later than 1990; these differences and causal stale settlement marks are explicit in the coverage table. Raw futures P&L is expressed relative to each contract's local-currency price; a USD conversion overlay and integer-lot sizing are outside this reproduction. No unavailable Pinnacle values are imputed.


## Data coverage

The full Appendix A mapping is in `configs/universe.csv`. Raw downloads and manifests are local and excluded from version control. A later first date below describes the selected WRDS contract series, not proof that no earlier data exists in any WRDS product.

In [ ]:
coverage = pd.read_csv(T/'data_coverage.csv')
display(coverage[['ticker','feature_days','first','last','rolls','stale_mark_days','calendar_reference_fills','equivalence']])

## Completed expanding-window experiments

All hyperparameter choices use pre-test validation observations. The final 2020-only test is separate from the five-window exhibits. `fit_seconds` is elapsed process time and includes user-requested pauses; it is not active compute time.

In [ ]:
inventory = pd.read_csv(T/'experiment_inventory.csv')
display(inventory)
display(pd.read_csv(T/'cpd_diagnostics.csv'))
assert not manifest['missing_experiments'], manifest['missing_experiments']

## Exhibit 3 — Raw signal output

Metrics below come from saved out-of-sample positions. Percentage columns use fractional units (0.15 = 15%).

In [ ]:
exhibit3 = pd.read_csv(T/'exhibit_3.csv').set_index('strategy')
display(exhibit3)

## Exhibit 4 — Rescaled to 15% volatility

This follows the author aggregation code: a single ex-post factor per strategy targets 15% mean main-window volatility; CAGR and drawdown are recomputed on the scaled daily returns. It is not a tradable volatility forecast.

In [ ]:
display(pd.read_csv(T/'exhibit_4.csv').set_index('strategy'))
display(pd.read_csv(T/'volatility_rescaling.csv'))
print('Separate linear display approximation; these are transformed summary metrics, not recomputed strategy CAGR/MDD:')
display(pd.read_csv(T/'exhibit_4_linear_display.csv').set_index('strategy'))
display(Image(filename=str(F/'wealth_comparison.png')))

## Comparison with the published study

The 50-candidate search budget matches the paper. Data source, historical coverage, roll construction, random-search realization and numerical backend differ. A failure to recover the published improvement is a substantive empirical outcome.

In [ ]:
comparison = pd.read_csv(T/'published_comparison.csv')
display(comparison[['strategy','sharpe_published','sharpe_reproduced','sharpe_difference','test_windows','status']])
display(Image(filename=str(F/'published_comparison.png')))
display(Image(filename=str(F/'cpd_window_sensitivity.png')))
display(pd.read_csv(T/'selected_cpd_windows.csv'))

## Appendix C — Transaction costs

Costs are in one-way bps of traded notional. Position reversal, volatility resizing, initial entry and final exit are charged. As in Appendix C, the core table uses changes in scaled exposure; the extension adds both futures-roll legs. The vertical line marks the professor's 2.5-bps scenario. Costs are hypothetical stress levels, not measured liquidity or impact estimates.

In [ ]:
costs = pd.read_csv(T/'transaction_costs.csv')
display(costs.pivot(index='strategy',columns='cost_bps',values='sharpe'))
display(Image(filename=str(F/'transaction_costs.png')))

## Futures rolls and execution sensitivity

Return endpoints always belong to the same contract. On a roll, the near/far settlement gap creates no cash flow. Both old and new traded exposures pay costs. The raw-return and delayed-signal scenarios below use the same already-fitted positions, without retraining on test data. Bid/ask spreads, market impact and integer-lot sizing are not identified by daily settlements.

In [ ]:
roll = pd.read_csv(E/'roll_costs.csv')
display(roll[roll.cost_bps.eq(2.5)][['strategy','scenario','sharpe','annual_return','max_drawdown','annual_turnover','annual_roll_extra']])
display(pd.read_csv(E/'break_even_costs.csv'))
display(Image(filename=str(ROOT/'results/extension/figures/roll_and_execution_costs.png')))

## Risk and regime sensitivity

The 2020 crisis is displayed separately. Position traces use frozen OOS forecasts. A paired 63-session moving-block bootstrap is conditional on the fitted models and does not quantify retraining uncertainty.

In [ ]:
display(Image(filename=str(F/'regime_sensitivity.png')))
display(Image(filename=str(F/'drawdowns.png')))
display(pd.read_csv(E/'paired_sharpe_bootstrap.csv'))
display(Image(filename=str(F/'position_diagnostics.png')))
display(pd.read_csv(T/'annual_metrics.csv').query('year == 2020'))

## Conclusions and limitations

In [ ]:
display(Markdown(conclusion_text()))

Repeated-seed significance has not been evaluated. Pinnacle's original histories and exact session/roll construction are unavailable in the connected sources. The daily close-t execution assumption is optimistic; the extension reports delayed-signal sensitivity. Fixed bps cannot establish implementable capacity, and MACD usefulness is not independently established by this reproduction. The five-window averaging convention and separate 2020 endpoint are fully specified above.

## Reproduction and provenance

From the prepared repository environment: `make reproduce`. For a new model fit from random initialization: `make fresh`. The final notebook loads reusable module outputs; it is not the source of experiment logic.

Sources: [original paper](https://arxiv.org/abs/2105.13727), [author GP/feature code](https://github.com/kieranjwood/slow-momentum-fast-reversion), [supporting author LSTM code](https://github.com/kieranjwood/trading-momentum-transformer). Preserved MIT licenses and source hashes are recorded in `UPSTREAM.json`. Course requirements and graded feedback were read from the supplied reference folder.

The report manifest records completed runs and feature/configuration hashes. Licensed market data and local credentials are excluded from Git.

In [ ]:
print(json.dumps({k:v for k,v in manifest.items() if k not in ['config']}, indent=2))